# AeroFleet — 1,000-case mass forensics run on Kaggle (free GPU/CPU)

Runs the real (non-mock) LLM evaluation harness against Kaggle, chunked across sessions using
the harness's own per-case checkpoint (`results.jsonl`). Safe to stop and resume — each session
picks up exactly where the last one left off.

This version runs the harness as a **background process** instead of a blocking cell, so you can
check progress / logs at any time without waiting, and stop it cleanly.

## Required one-time manual setup (Kaggle UI, not this notebook)

1. **Settings → Internet → On.** GPU accelerator is optional now — the model (3.8B, ~3.2GB) runs
   fine on CPU alone, so a plain CPU session works too if GPU quota is tight.
2. **Add-ons → Secrets → add a secret named `GH_PAT`**: a GitHub fine-grained Personal Access
   Token, read-only, scoped to just this repo (`AdityaPathare46/aerofleet`, private). Toggle it
   **on** for this notebook specifically — creating the secret alone isn't enough.
3. Session cap is several hours. When a session ends, **Save Version** (use the option that saves
   your *current* session state, not the one that re-runs everything from scratch) before closing
   — that persists `/kaggle/working` (including `results.jsonl`) as this notebook's Output for
   next session to resume from.

## Splitting the work across machines

`--target 1000 --batch-size 25` gives 40 batches. This notebook is scoped to **batches 1-14**
(`ONLY_BATCHES` in the run cell) so it can run in parallel with
`research/colab_mass_forensics_run.ipynb` (batches 27-40) and a third machine (batches 15-26)
without redoing each other's work:

| Machine | Batches | Cases |
|---|---|---|
| **This Kaggle notebook** | **1-14** | **MFI-00001 – MFI-00350** |
| Colab | 27-40 | MFI-00651 – MFI-01000 |
| Third machine | 15-26 | MFI-00351 – MFI-00650 |

Running solo? Set `ONLY_BATCHES = None` in the run cell instead.

## Model roster

Every agent shares one model — `phi4-mini-reasoning` (Microsoft, 3.8B, ~3.2GB, verified real and
pullable) — chosen to run on ordinary hardware without the VRAM-thrashing a larger, multi-model
roster caused on constrained GPUs. **Every machine in the split above runs the identical real
roster** — no per-machine model substitution, no methodology deviation to track.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU visible — that's fine, phi4-mini-reasoning runs on CPU."


## 1. Clone the private repo

Uses the `GH_PAT` secret — the token is interpolated into the clone URL and never printed or
stored in this notebook.


In [ ]:
from kaggle_secrets import UserSecretsClient
_token = UserSecretsClient().get_secret("GH_PAT")

REPO_DIR = "/kaggle/working/aerofleet"
import os
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://{_token}@github.com/AdityaPathare46/aerofleet.git {REPO_DIR}
else:
    print("Repo already present — pulling latest instead of a fresh clone.")
    !cd {REPO_DIR} && git pull

del _token  # don't leave it bound in the kernel namespace longer than needed
%cd {REPO_DIR}


In [ ]:
!pip install -q -r requirements.txt


## 2. Install Ollama and start the server in this session

`OLLAMA_KEEP_ALIVE=-1` tells Ollama to never voluntarily unload the model due to idle time —
defensive, though with a single ~3.2GB model there's no other model competing for space so this
mostly just avoids an unnecessary reload if calls are spaced apart.


In [ ]:
# zstd: required by the Ollama installer to extract its archive.
# pciutils (lspci): lets the installer auto-detect the GPU and install the
# matching CUDA runtime — without it, install silently warns and may fall
# back to a CPU-only build, which would make the run far too slow to finish.
!apt-get update -qq && apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import subprocess, time, requests, os

os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

log = open("/kaggle/working/ollama_serve.log", "a")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())

for _ in range(30):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=5)
        print("Ollama server is up.")
        break
    except requests.exceptions.RequestException:
        time.sleep(2)
else:
    raise RuntimeError("Ollama server did not come up — check /kaggle/working/ollama_serve.log")

time.sleep(2)
!grep -i -E "gpu|cuda|library=" /kaggle/working/ollama_serve.log | tail -5


## 3. Pull the model

One model now (~3.2GB), not four — a couple of minutes, not tens of GB. Re-run this cell if
interrupted; Ollama resumes partial downloads.


In [ ]:
print("--- pulling phi4-mini-reasoning ---")
!ollama pull phi4-mini-reasoning
!ollama list


## 4. Resume from a previous session's checkpoint (if any)

If you saved a prior session's output and re-attached it as an input dataset (Add Data → Your
Work → this notebook's earlier Output), point `PREVIOUS_RESULTS_DIR` at the mounted path Kaggle
shows in the file browser (something like `/kaggle/input/<notebook-slug>/mass_forensics_kaggle`) —
its `results.jsonl` gets copied into this session's study dir so the harness resumes instead of
starting over. Leave as `None` for the very first session.


In [ ]:
import shutil
from pathlib import Path

STUDY_DIR = Path("/kaggle/working/mass_forensics_kaggle")
STUDY_DIR.mkdir(parents=True, exist_ok=True)

PREVIOUS_RESULTS_DIR = None  # e.g. "/kaggle/input/<notebook-slug>/mass_forensics_kaggle"

if PREVIOUS_RESULTS_DIR is not None:
    prev = Path(PREVIOUS_RESULTS_DIR)
    for fname in ["results.jsonl", "run_config.json", "dataset_manifest.json"]:
        src = prev / fname
        if src.exists():
            shutil.copy(src, STUDY_DIR / fname)
            print(f"Restored {fname} from previous session")
        else:
            print(f"{fname} not found at {src} — check PREVIOUS_RESULTS_DIR is correct")
else:
    print("No previous session pointed to — starting a fresh study.")


## 5. Record this run's configuration

No per-agent model overrides needed anymore — every agent already shares the same
`phi4-mini-reasoning` model via `AgentFactory.DEFAULT_MODEL_MAP`. This cell just records the
run's metadata for traceability.


In [ ]:
import os, json, subprocess, datetime

os.environ["OLLAMA_HOST"] = "http://localhost:11434"
os.environ.pop("USE_MOCK_AGENTS", None)  # make sure we are NOT in mock mode

try:
    gpu_name = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True
    ).stdout.strip() or "CPU (no GPU visible)"
except FileNotFoundError:
    gpu_name = "CPU (no GPU visible)"

metadata = {
    "platform": "kaggle",
    "gpu": gpu_name,
    "only_batches": "1-14",
    "run_started_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "model_roster": {
        "shared_model": "phi4-mini-reasoning",
        "note": "every agent uses this same model — no per-machine substitution needed",
    },
}
with open(STUDY_DIR / "cloud_run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))


## 6. Start the harness — runs in the background

This does NOT block the notebook — it starts the run and returns immediately. Use the cells below
to check on it, any time, as often as you like, without needing to stop it first.


In [ ]:
import subprocess, os, requests

# Confirm Ollama is actually reachable BEFORE starting the harness — this exact check would
# have caught a real incident where 350 cases in a row failed instantly because Ollama simply
# wasn't running when the harness started.
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    print("Ollama is reachable:", r.json())
except requests.exceptions.RequestException as e:
    raise RuntimeError(
        f"Ollama is NOT reachable ({e}). Do not start the harness yet — go back and re-run "
        "the 'start server' cell first, then re-run this cell."
    )

ONLY_BATCHES = "1-14"  # this machine's assigned, non-overlapping slice — see the table above.
# Running solo? Set ONLY_BATCHES = None instead.

harness_log_path = STUDY_DIR / "harness_run.log"
harness_log = open(harness_log_path, "a")

cmd = [
    "python", "-m", "scenario_engine.mass_forensics_evaluation",
    "--study-dir", str(STUDY_DIR), "--target", "1000", "--batch-size", "25",
    "--retry-failed-max", "5",  # a bit more forgiving of a transient connection hiccup
]
if ONLY_BATCHES:
    cmd += ["--only-batches", ONLY_BATCHES]

harness_proc = subprocess.Popen(cmd, stdout=harness_log, stderr=subprocess.STDOUT, env=os.environ.copy())
print(f"Harness started in the background, PID {harness_proc.pid}.")
print(f"Logging to {harness_log_path}")
print("Run the cells below any time to check progress — no need to wait or stop this first.")


## 7. Monitoring — run any of these any time, as often as you like


In [ ]:
# Tail the live log
!tail -n 40 {harness_log_path}


In [ ]:
# Is it still running? (None = still running; a number = exit code, it finished/stopped)
print("exit code (None = still running):", harness_proc.poll())


In [ ]:
# How many case-attempts recorded so far
results_file = STUDY_DIR / "results.jsonl"
if results_file.exists():
    with open(results_file) as f:
        n = sum(1 for _ in f)
    print(f"{n} case-attempts recorded so far (target: 1000 cases; retries count as separate attempts).")
else:
    print("No results yet.")


In [ ]:
# Thrashing check: if this shows a model, then a few cells later shows a DIFFERENT model
# (or nothing) repeatedly, models are being evicted and reloaded between calls — worth flagging.
!ollama ps


## 8. Stopping cleanly

Use this instead of any Kaggle "stop cell" UI button — it terminates only the harness process,
leaving the Ollama server running undisturbed (a UI-level interrupt may not make that distinction
and can take the server down with it, which happened in an earlier session).


In [ ]:
harness_proc.terminate()
harness_proc.wait(timeout=30)
print("Harness stopped cleanly. results.jsonl has everything completed up to this point.")


## 9. End of session — persist for next time

Click **Save Version** (use the option that saves your *current* session state — not the one that
re-runs every cell from scratch, which would restart the whole setup). That snapshots everything
under `/kaggle/working` (including `results.jsonl` and `cloud_run_metadata.json`) as this
notebook's Output. Next session: **Add Data → Your Work → (this notebook, latest version)**, then
set `PREVIOUS_RESULTS_DIR` in the resume cell to the mounted path Kaggle shows for it, and re-run
from the top. Repeat until this machine's assigned batch range (1-14) is complete.

## 10. Once all machines are done (or you want an interim pooled check)

Pull each machine's `results.jsonl` + `dataset_manifest.json` + `run_config.json` to one place and
run `research/merge_batched_results.py` to pool them, then `--report-only` on the merged study —
see `research/colab_mass_forensics_run.ipynb`'s final cell for the exact commands.
